# 6.1 全参数微调 (Full Fine-Tuning)

全参数微调是微调的最基本形式，解冻模型所有参数在目标任务数据上进行训练，是大模型适配特定领域的核心方法。

本节涵盖：
- 显存分析与优化
- 训练配置最佳实践
- 灾难性遗忘与缓解
- 梯度检查点
- 数据质量与训练监控

## 1. 显存分析与优化

**目的**：理解全参数微调的显存需求，掌握优化策略

**显存组成**（以7B模型为例，AdamW优化器，FP16/BF16训练）：
- **模型参数**：7B × 2字节 = 14 GB
- **梯度**：7B × 2字节 = 14 GB
- **优化器状态**：7B × (4+4+4)字节 = 84 GB（FP32主参数+动量m+方差v）
- **激活值**：取决于batch_size和seq_len，通常5-15 GB
- **总计**：约120-130 GB（单GPU无法容纳，需要分布式训练）

**关键优化策略**：
- **混合精度训练**：前向/反向用FP16/BF16，优化器用FP32，节省约50%激活值显存
- **梯度检查点**：用时间换空间，只保存部分激活值，需要时重新计算，节省60-70%激活值显存
- **梯度累积**：用小batch多次前向/反向累积梯度，等效大batch训练
- **ZeRO优化**：将优化器状态/梯度/参数分片到多GPU，显著降低单卡显存需求

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def estimate_memory(n_params_b, seq_len, batch_size, hidden_dim, n_layers,
                   precision='bf16', optimizer='adamw', grad_checkpoint=False,
                   zero_stage=0, n_gpus=1):
    bytes_per_param = 2 if precision in ('fp16', 'bf16') else 4
    param_mem = n_params_b * 1e9 * bytes_per_param
    grad_mem = param_mem

    if optimizer == 'adamw':
        optimizer_mem = n_params_b * 1e9 * 12
    else:
        optimizer_mem = n_params_b * 1e9 * 8

    activation_per_token = hidden_dim * n_layers * 4 * bytes_per_param
    if grad_checkpoint:
        activation_per_token *= 0.3
    activation_mem = activation_per_token * seq_len * batch_size

    total = param_mem + grad_mem + optimizer_mem + activation_mem

    if zero_stage >= 1:
        optimizer_mem /= n_gpus
    if zero_stage >= 2:
        grad_mem /= n_gpus
    if zero_stage >= 3:
        param_mem /= n_gpus

    per_gpu = param_mem + grad_mem + optimizer_mem + activation_mem
    return {'total_gb': total / 1e9, 'per_gpu_gb': per_gpu / 1e9,
            'params_gb': param_mem / 1e9, 'grads_gb': grad_mem / 1e9,
            'optimizer_gb': optimizer_mem / 1e9, 'activations_gb': activation_mem / 1e9}

print('=== Full Fine-Tuning Memory Analysis ===')
print(f'\n--- 7B Model (LLaMA-2-7B style) ---')
configs = [
    ('Baseline (1xA100)', dict(n_params_b=7, seq_len=2048, batch_size=1, hidden_dim=4096, n_layers=32, zero_stage=0, n_gpus=1, grad_checkpoint=False)),
    ('+Grad Ckpt',        dict(n_params_b=7, seq_len=2048, batch_size=1, hidden_dim=4096, n_layers=32, zero_stage=0, n_gpus=1, grad_checkpoint=True)),
    ('+ZeRO-1 (8xGPU)',   dict(n_params_b=7, seq_len=2048, batch_size=1, hidden_dim=4096, n_layers=32, zero_stage=1, n_gpus=8, grad_checkpoint=True)),
    ('+ZeRO-2 (8xGPU)',   dict(n_params_b=7, seq_len=2048, batch_size=1, hidden_dim=4096, n_layers=32, zero_stage=2, n_gpus=8, grad_checkpoint=True)),
    ('+ZeRO-3 (8xGPU)',   dict(n_params_b=7, seq_len=2048, batch_size=1, hidden_dim=4096, n_layers=32, zero_stage=3, n_gpus=8, grad_checkpoint=True)),
]
for name, cfg in configs:
    result = estimate_memory(**cfg)
    print(f'  {name:20s}: per_gpu={result["per_gpu_gb"]:.1f}GB, '
          f'params={result["params_gb"]:.1f}GB, grads={result["grads_gb"]:.1f}GB, '
          f'opt={result["optimizer_gb"]:.1f}GB, act={result["activations_gb"]:.1f}GB')

print(f'\n--- 70B Model (LLaMA-2-70B style) ---')
configs_70b = [
    ('ZeRO-3 (8xA100)',   dict(n_params_b=70, seq_len=2048, batch_size=1, hidden_dim=8192, n_layers=80, zero_stage=3, n_gpus=8, grad_checkpoint=True)),
    ('ZeRO-3 (64xA100)',  dict(n_params_b=70, seq_len=2048, batch_size=1, hidden_dim=8192, n_layers=80, zero_stage=3, n_gpus=64, grad_checkpoint=True)),
]
for name, cfg in configs_70b:
    result = estimate_memory(**cfg)
    print(f'  {name:20s}: per_gpu={result["per_gpu_gb"]:.1f}GB')

print(f'\nKey: 7B model needs ZeRO-2/3 + gradient checkpointing to fit on 8xA100-80GB.')
print(f'70B model requires 64+ GPUs with ZeRO-3 for full fine-tuning.')

## 2. 训练配置最佳实践

**目的**：掌握全参数微调的关键超参数配置

**核心超参数**：
- **学习率**：1e-5 ~ 5e-5，比预训练小10-100倍，过大会导致灾难性遗忘
- **Epoch数**：通常2-5个epoch，过多会过拟合
- **Batch Size**：等效batch 64-256，通过梯度累积实现
- **Warmup**：总步数的3-10%
- **权重衰减**：0.01-0.1，bias和LayerNorm参数不施加
- **LR调度**：余弦衰减到峰值的10%

**与预训练的区别**：
- 预训练：lr=3e-4, 大batch, 多epoch不重复数据
- 微调：lr=2e-5, 小batch, 少epoch, 数据可重复

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class SimpleTransformer(nn.Module):
    def __init__(self, d_model=256, n_heads=4, n_layers=4, vocab_size=1000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(512, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
                                                     dim_feedforward=d_model*4,
                                                     batch_first=True, dropout=0.1)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, labels=None):
        seq_len = input_ids.size(1)
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        h = self.embedding(input_ids) + self.pos_embedding(positions)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=input_ids.device)
        h = self.transformer(h, mask=causal_mask)
        logits = self.lm_head(h)
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, logits.size(-1)),
                                   shift_labels.view(-1), ignore_index=-100)
            return logits, loss
        return logits, None

model = SimpleTransformer(d_model=256, n_heads=4, n_layers=4, vocab_size=1000)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

decay_params, no_decay_params = [], []
for name, param in model.named_parameters():
    if 'bias' in name or 'LayerNorm' in name or 'norm' in name:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW([
    {'params': decay_params, 'weight_decay': 0.01, 'lr': 2e-5},
    {'params': no_decay_params, 'weight_decay': 0.0, 'lr': 2e-5},
], lr=2e-5, betas=(0.9, 0.999))

n_data = 500
seq_len = 64
input_ids = torch.randint(0, 1000, (n_data, seq_len))
labels = input_ids.clone()
mask = torch.rand(input_ids.shape) < 0.15
labels[mask] = -100

print('\n=== Full Fine-Tuning Training Loop ===')
batch_size = 16
accumulation_steps = 4
effective_batch = batch_size * accumulation_steps
n_epochs = 3
warmup_ratio = 0.05
total_steps = (n_data // batch_size) * n_epochs
warmup_steps = int(total_steps * warmup_ratio)

def get_lr(step, warmup, peak_lr, total):
    if step < warmup:
        return peak_lr * step / warmup
    progress = (step - warmup) / (total - warmup)
    return peak_lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress)))

step = 0
for epoch in range(n_epochs):
    indices = torch.randperm(n_data)
    epoch_loss = 0.0
    n_batches = 0

    for i in range(0, n_data, batch_size):
        batch_idx = indices[i:i+batch_size]
        X = input_ids[batch_idx]
        Y = labels[batch_idx]

        _, loss = model(X, Y)
        loss = loss / accumulation_steps
        loss.backward()

        if (i // batch_size + 1) % accumulation_steps == 0:
            lr = get_lr(step, warmup_steps, 2e-5, total_steps)
            for pg in optimizer.param_groups:
                pg['lr'] = lr
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            step += 1

        epoch_loss += loss.item() * accumulation_steps
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    print(f'Epoch {epoch+1}: avg_loss={avg_loss:.4f}, lr={get_lr(step, warmup_steps, 2e-5, total_steps):.2e}')

print(f'\nTraining config:')
print(f'  Effective batch: {effective_batch} = {batch_size} × {accumulation_steps}')
print(f'  Total steps: {total_steps}, Warmup: {warmup_steps}')
print(f'  Peak LR: 2e-5, Min LR: 2e-6 (10% of peak)')
print(f'  Weight decay: 0.01 (excluded for bias/LayerNorm)')
print(f'  Grad clip: 1.0')
print(f'\nKey: Full fine-tuning uses much smaller LR than pretraining (2e-5 vs 3e-4).')
print(f'Gradient accumulation + clipping + cosine decay are standard practices.')

## 3. 灾难性遗忘与缓解

**目的**：在全参数微调中保留预训练知识

**问题**：全参数微调会修改模型所有参数，可能导致模型在微调数据上表现好但在通用任务上退化（灾难性遗忘）。

**缓解策略**：
- **学习率控制**：使用较小的学习率（1e-5~5e-5），减少参数偏移
- **L2正则化/权重衰减**：限制参数偏离预训练值
- **EWC（Elastic Weight Consolidation）**：对重要参数施加更强的正则化
- **数据混合**：在微调数据中混入少量预训练数据
- **Early Stopping**：在验证集性能开始下降时停止训练

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SimpleModel(nn.Module):
    def __init__(self, d=64, n_classes=5):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Linear(128, 64))
        self.head = nn.Linear(64, n_classes)
    def forward(self, x):
        return self.head(self.encoder(x))

class EWCRegularizer:
    def __init__(self, model, dataloader, n_samples=50, lambda_ewc=1000):
        self.model = model
        self.lambda_ewc = lambda_ewc
        self.fisher_info = {n: torch.zeros_like(p) for n, p in model.named_parameters()}
        self.optimal_params = {n: p.clone() for n, p in model.named_parameters()}

        model.eval()
        for X, y in dataloader:
            model.zero_grad()
            loss = F.cross_entropy(model(X), y)
            loss.backward()
            for n, p in model.named_parameters():
                if p.grad is not None:
                    self.fisher_info[n] += p.grad.data.pow(2) / n_samples

    def penalty(self, model):
        loss = 0
        for n, p in model.named_parameters():
            loss += (self.fisher_info[n] * (p - self.optimal_params[n]).pow(2)).sum()
        return self.lambda_ewc * loss

pretrain_model = SimpleModel()
X_pretrain = torch.randn(200, 64)
y_pretrain = torch.randint(0, 5, (200,))
pretrain_opt = torch.optim.Adam(pretrain_model.parameters(), lr=1e-3)
for _ in range(50):
    loss = F.cross_entropy(pretrain_model(X_pretrain), y_pretrain)
    pretrain_opt.zero_grad(); loss.backward(); pretrain_opt.step()

pretrain_acc = (pretrain_model(X_pretrain).argmax(-1) == y_pretrain).float().mean()
print(f'Pretrained model accuracy on original task: {pretrain_acc:.3f}')

ewc = EWCRegularizer(pretrain_model, [(X_pretrain, y_pretrain)], lambda_ewc=500)

X_finetune = torch.randn(100, 64) + 3.0
y_finetune = torch.randint(0, 5, (100,))

model_naive = SimpleModel()
model_naive.load_state_dict(pretrain_model.state_dict())
model_ewc = SimpleModel()
model_ewc.load_state_dict(pretrain_model.state_dict())
model_l2 = SimpleModel()
model_l2.load_state_dict(pretrain_model.state_dict())

ref_params = {n: p.clone() for n, p in pretrain_model.named_parameters()}

opt_naive = torch.optim.Adam(model_naive.parameters(), lr=1e-3)
opt_ewc = torch.optim.Adam(model_ewc.parameters(), lr=1e-3)
opt_l2 = torch.optim.Adam(model_l2.parameters(), lr=1e-3)

print('\n=== Catastrophic Forgetting Mitigation ===')
for step in range(30):
    loss_naive = F.cross_entropy(model_naive(X_finetune), y_finetune)
    opt_naive.zero_grad(); loss_naive.backward(); opt_naive.step()

    loss_ewc = F.cross_entropy(model_ewc(X_finetune), y_finetune) + ewc.penalty(model_ewc)
    opt_ewc.zero_grad(); loss_ewc.backward(); opt_ewc.step()

    loss_l2 = F.cross_entropy(model_l2(X_finetune), y_finetune)
    l2_reg = sum((p - ref_params[n]).pow(2).sum() for n, p in model_l2.named_parameters())
    loss_l2 = loss_l2 + 0.01 * l2_reg
    opt_l2.zero_grad(); loss_l2.backward(); opt_l2.step()

    if (step + 1) % 10 == 0:
        acc_naive_orig = (model_naive(X_pretrain).argmax(-1) == y_pretrain).float().mean()
        acc_ewc_orig = (model_ewc(X_pretrain).argmax(-1) == y_pretrain).float().mean()
        acc_l2_orig = (model_l2(X_pretrain).argmax(-1) == y_pretrain).float().mean()
        acc_naive_new = (model_naive(X_finetune).argmax(-1) == y_finetune).float().mean()
        acc_ewc_new = (model_ewc(X_finetune).argmax(-1) == y_finetune).float().mean()
        acc_l2_new = (model_l2(X_finetune).argmax(-1) == y_finetune).float().mean()
        print(f'Step {step+1}:')
        print(f'  Naive:  orig={acc_naive_orig:.3f}, new={acc_naive_new:.3f}')
        print(f'  L2:     orig={acc_l2_orig:.3f}, new={acc_l2_new:.3f}')
        print(f'  EWC:    orig={acc_ewc_orig:.3f}, new={acc_ewc_new:.3f}')

print(f'\nKey: Naive fine-tuning forgets the original task (low orig accuracy).')
print(f'EWC preserves important parameters by penalizing changes proportional to Fisher information.')
print(f'L2 regularization is simpler but less targeted than EWC.')

## 4. 梯度检查点

**目的**：用计算时间换取显存空间，使大模型微调成为可能

**基本原理**：正常训练时，前向传播保存所有中间激活值供反向传播使用。梯度检查点只保存部分层的激活值（检查点），反向传播需要时重新计算被丢弃的激活值。

**效果**：
- 显存降低：激活值显存从O(n)降至O(√n)，n为层数
- 速度代价：前向计算量增加约33%（需要重新计算丢弃的激活值）
- 典型场景：7B模型微调时，梯度检查点可将激活值显存从15GB降至5GB

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

torch.manual_seed(42)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True, dropout=dropout)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model*4), nn.GELU(), nn.Linear(d_model*4, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0])
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x

class TransformerModel(nn.Module):
    def __init__(self, d_model=256, n_heads=4, n_layers=8, vocab_size=1000, use_grad_ckpt=False):
        super().__init__()
        self.use_grad_ckpt = use_grad_ckpt
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([TransformerBlock(d_model, n_heads) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        h = self.embedding(input_ids)
        for layer in self.layers:
            if self.use_grad_ckpt and self.training:
                h = torch.utils.checkpoint.checkpoint(layer, h, use_reentrant=False)
            else:
                h = layer(h)
        h = self.norm(h)
        return self.head(h)

vocab_size, seq_len, batch_size = 1000, 128, 8
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
labels = input_ids.clone()

print('=== Gradient Checkpointing Comparison ===')

model_no_ckpt = TransformerModel(d_model=256, n_heads=4, n_layers=8, use_grad_ckpt=False)
model_with_ckpt = TransformerModel(d_model=256, n_heads=4, n_layers=8, use_grad_ckpt=True)
model_with_ckpt.load_state_dict(model_no_ckpt.state_dict())

n_params = sum(p.numel() for p in model_no_ckpt.parameters())
print(f'Model parameters: {n_params:,}')

torch.cuda.empty_cache() if torch.cuda.is_available() else None

model_no_ckpt.train()
start = time.time()
logits = model_no_ckpt(input_ids)
loss = F.cross_entropy(logits[:, :-1].reshape(-1, vocab_size), labels[:, 1:].reshape(-1))
loss.backward()
time_no_ckpt = time.time() - start
mem_no_ckpt = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0

model_with_ckpt.train()
start = time.time()
logits2 = model_with_ckpt(input_ids)
loss2 = F.cross_entropy(logits2[:, :-1].reshape(-1, vocab_size), labels[:, 1:].reshape(-1))
loss2.backward()
time_with_ckpt = time.time() - start
mem_with_ckpt = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0

print(f'\nWithout grad checkpointing:')
print(f'  Time: {time_no_ckpt:.3f}s')
print(f'  GPU memory: {mem_no_ckpt:.2f}GB' if torch.cuda.is_available() else '  GPU memory: N/A (CPU only)')

print(f'\nWith grad checkpointing:')
print(f'  Time: {time_with_ckpt:.3f}s ({time_with_ckpt/time_no_ckpt:.2f}x)')
print(f'  GPU memory: {mem_with_ckpt:.2f}GB' if torch.cuda.is_available() else '  GPU memory: N/A (CPU only)')

print(f'\nKey: Gradient checkpointing trades ~33% more compute for 60-70% less activation memory.')
print(f'Essential for fine-tuning models >7B on limited GPU memory.')
print(f'Usage: torch.utils.checkpoint.checkpoint(layer, x, use_reentrant=False)')

## 5. 数据质量与训练监控

**目的**：确保微调数据质量和训练过程稳定

**数据质量关键指标**：
- **重复率**：高重复数据导致过拟合，应去重
- **长度分布**：过长序列浪费计算，过短序列信息不足
- **标签质量**：错误标签直接损害模型，需要人工抽检
- **数据多样性**：单一来源数据导致偏见

**训练监控指标**：
- **训练Loss**：应平稳下降，spike可能表明数据问题或学习率过大
- **验证Loss**：持续上升=过拟合，需要early stopping
- **梯度范数**：突然增大=训练不稳定，需要降低学习率
- **学习率**：确认调度器正常工作

**Loss Spike处理**：
- 跳过当前batch（可能是脏数据）
- 降低学习率
- 恢复到上一个checkpoint

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque

torch.manual_seed(42)

class TrainingMonitor:
    def __init__(self, patience=5, spike_threshold=3.0, window_size=10):
        self.patience = patience
        self.spike_threshold = spike_threshold
        self.window_size = window_size
        self.train_losses = []
        self.val_losses = []
        self.grad_norms = []
        self.best_val_loss = float('inf')
        self.patience_counter = 0
        self.spike_count = 0

    def log_train(self, loss, grad_norm=None):
        self.train_losses.append(loss)
        if grad_norm is not None:
            self.grad_norms.append(grad_norm)

        if len(self.train_losses) >= self.window_size:
            recent = self.train_losses[-self.window_size:]
            mean_loss = sum(recent) / len(recent)
            if loss > mean_loss * self.spike_threshold:
                self.spike_count += 1
                return 'spike'
        return 'ok'

    def log_val(self, loss):
        self.val_losses.append(loss)
        if loss < self.best_val_loss:
            self.best_val_loss = loss
            self.patience_counter = 0
            return 'best'
        else:
            self.patience_counter += 1
            if self.patience_counter >= self.patience:
                return 'stop'
            return 'worse'

    def should_save(self):
        return self.val_losses and self.val_losses[-1] <= self.best_val_loss

model = nn.Sequential(
    nn.Linear(64, 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 5)
)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
monitor = TrainingMonitor(patience=5, spike_threshold=2.5)

X_train = torch.randn(200, 64)
y_train = torch.randint(0, 5, (200,))
X_val = torch.randn(50, 64)
y_val = torch.randint(0, 5, (50,))

best_state = None
print('=== Training Monitoring Demo ===')
for epoch in range(30):
    model.train()
    batch_losses = []
    for i in range(0, 200, 20):
        loss = F.cross_entropy(model(X_train[i:i+20]), y_train[i:i+20])
        optimizer.zero_grad()
        loss.backward()
        grad_norm = torch.cat([p.grad.flatten() for p in model.parameters() if p.grad is not None]).norm().item()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        status = monitor.log_train(loss.item(), grad_norm)
        batch_losses.append(loss.item())
        if status == 'spike':
            print(f'  WARNING: Loss spike detected at epoch {epoch+1}!')

    model.eval()
    with torch.no_grad():
        val_loss = F.cross_entropy(model(X_val), y_val).item()
    val_status = monitor.log_val(val_loss)

    if val_status == 'best':
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if (epoch + 1) % 5 == 0:
        train_acc = (model(X_train).argmax(-1) == y_train).float().mean().item()
        val_acc = (model(X_val).argmax(-1) == y_val).float().mean().item()
        print(f'Epoch {epoch+1}: train_loss={sum(batch_losses)/len(batch_losses):.4f}, '
              f'val_loss={val_loss:.4f}, train_acc={train_acc:.3f}, val_acc={val_acc:.3f}, '
              f'best_val={monitor.best_val_loss:.4f}')

    if val_status == 'stop':
        print(f'\nEarly stopping at epoch {epoch+1}! No improvement for {monitor.patience} epochs.')
        break

if best_state:
    model.load_state_dict(best_state)
    print(f'\nRestored best model (val_loss={monitor.best_val_loss:.4f})')

print(f'\nLoss spikes detected: {monitor.spike_count}')
print(f'\nKey: Training monitoring is essential for production fine-tuning.')
print(f'Early stopping prevents overfitting; spike detection catches data/learning rate issues.')
print(f'Always save the best checkpoint, not the last one.')